# Week 6 Concepts — Web-Search-Augmented Research

This notebook mirrors the four daily notes, one section per day, with runnable code.

**What was actually executed and verified in this sandbox:** the mock search client, the
TF-IDF re-ranking example (real `scikit-learn` cosine-similarity scores), the citation
validator, the `research()` pipeline, and the tool-evaluation scoring function. The two
live-network cells (DuckDuckGo HTML search in Day 1, page fetch in Day 3) were also tested
successfully in this environment, but are wrapped in `try/except` with a fixed fallback since
a live web endpoint is not something to depend on being reachable every time this notebook
runs. The one cell that is **not executable here** is the `sentence-transformers` example
near the end of Day 2 — `torch`/`transformers`/`sentence-transformers` are not installed in
this sandbox. It is written against the real, current API and is included for reference only.

## Day 1: Search APIs for Fresh Information

A search API takes a text query and returns a list of results, each carrying a title, a
snippet, and a source URL. Below is a small **mock** search client — no network call, no API
key — that matches the shape a real client would have. Swapping the mock for a real client
later only means changing the body of `search_web`, not any of its callers.

In [ ]:
from dataclasses import dataclass

@dataclass
class SearchResult:
    title: str
    snippet: str
    source_url: str

# Mock local "index" standing in for a real search provider's backend.
# A real implementation would send `query` to an HTTP endpoint and parse JSON results instead.
_MOCK_INDEX: dict[str, list[SearchResult]] = {
    "launch window definition": [
        SearchResult(
            "Launch Windows Explained",
            "A launch window is the time period during which a rocket can lift off to reach its target orbit.",
            "https://example-space.org/launch-windows",
        ),
        SearchResult(
            "Orbital Mechanics 101",
            "Launch windows are constrained by the relative positions of Earth and the destination body.",
            "https://example-space.org/orbital-mechanics",
        ),
    ],
    "reusable rocket landing methods": [
        SearchResult(
            "Booster Recovery Techniques",
            "Modern boosters land using either a controlled powered descent or a parachute-assisted splashdown.",
            "https://example-space.org/booster-recovery",
        ),
    ],
}

def search_web_stub(query: str, max_results: int = 5) -> list[SearchResult]:
    """Mock search client. A real one would call a provider API and map its JSON to SearchResult."""
    key = query.lower().strip()
    return _MOCK_INDEX.get(key, [])[:max_results]  # -> list[SearchResult], len 0..max_results

# A specific query hits the mock index; a vague one returns nothing useful.
# Against a REAL search engine, the vague query wouldn't return nothing -- it would return
# many results that are technically on-topic but useless for this specific question.
good_hits = search_web_stub("launch window definition")
vague_hits = search_web_stub("space stuff")
print(f"specific query -> {len(good_hits)} results")
print(f"vague query    -> {len(vague_hits)} results")

In [ ]:
def build_context_bundle(results: list[SearchResult]) -> str:
    """Combine search results into one string for an LLM prompt, numbering each one so a
    later citation like '[2]' can be traced straight back to results[1]. Keeping the URL
    attached to every block is what makes citation possible downstream (Day 3)."""
    blocks = []
    for i, r in enumerate(results, start=1):
        blocks.append(f"[{i}] {r.title}\n{r.snippet}\nSource: {r.source_url}")
    return "\n\n".join(blocks)  # -> str, one blank-line-separated block per SearchResult

bundle = build_context_bundle(good_hits)
print(bundle)

### A real call, for comparison

Everything above is deterministic and offline. Here is the same `SearchResult` shape produced
by an actual HTTP request against DuckDuckGo's public HTML results page (no API key) — useful
for seeing the real request/parse mechanics that any paid search API is doing under the hood.
Wrapped in `try/except` with a fixed fallback so this cell still runs with no network access.

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse, parse_qs, unquote

def search_web_live(query: str, max_results: int = 5) -> list[SearchResult]:
    """Illustrative only -- hits an undocumented HTML endpoint with no API key, so it is
    fragile (markup can change) and not appropriate for production use. A real integration
    should call a documented, ToS-compliant provider (Tavily, Brave Search API, Bing Web
    Search, Serper, etc.) with a proper key instead of scraping HTML."""
    resp = requests.get(
        "https://html.duckduckgo.com/html/",
        params={"q": query},
        headers={"User-Agent": "Mozilla/5.0"},
        timeout=10,
    )
    soup = BeautifulSoup(resp.text, "html.parser")
    results: list[SearchResult] = []
    for row in soup.select(".result")[:max_results]:
        link = row.select_one(".result__title a")
        snippet_el = row.select_one(".result__snippet")
        if link is None:
            continue
        # The HTML endpoint wraps the real destination in a `/l/?uddg=<url-encoded-url>`
        # redirect link rather than linking to it directly -- decode that back to the
        # actual source URL so citations later point somewhere real.
        qs = parse_qs(urlparse(link.get("href", "")).query)
        real_url = unquote(qs["uddg"][0]) if "uddg" in qs else link.get("href", "")
        results.append(SearchResult(
            title=link.get_text(strip=True),
            snippet=snippet_el.get_text(strip=True) if snippet_el else "",
            source_url=real_url,
        ))
    return results  # -> list[SearchResult], len <= max_results

# Fixed fallback fixture: exactly what search_web_live("python asyncio wait_for timeout")
# returned when this was captured, so the cell still demonstrates the real shape offline.
_LIVE_SEARCH_FALLBACK = [
    SearchResult(
        "Asyncio wait_for() to Wait With a Timeout - SuperFastPython",
        "",
        "https://superfastpython.com/asyncio-wait_for/",
    ),
    SearchResult(
        "Python asyncio.wait_for(): Cancel a Task with a Timeout",
        "",
        "https://www.pythontutorial.net/python-concurrency/python-asyncio-wait_for/",
    ),
]

try:
    live_hits = search_web_live("python asyncio wait_for timeout", max_results=5)
    if not live_hits:
        raise ValueError("live search returned no results")
    source_label = "LIVE"
except Exception as exc:
    live_hits = _LIVE_SEARCH_FALLBACK
    source_label = f"FALLBACK (live call unavailable: {type(exc).__name__})"

print(f"[{source_label}] {len(live_hits)} results")
for r in live_hits[:3]:
    print(f"  - {r.title}  |  {r.source_url}")

## Day 2: Embedding-Based Re-Ranking of Search Results

A search API's own ranking optimizes for general popularity, not the exact query your agent
asked. Re-ranking re-scores raw hits against that specific query using cosine similarity over
embedded text, so only the genuinely relevant results survive to reach the LLM.

In [ ]:
import hashlib
import re
from collections import Counter

VECTOR_SIZE = 64

def embed_text(text: str) -> list[float]:
    """Deterministic, fully local stand-in for a real embedding model call. Every word
    hashes to the same bucket every time, so identical text always produces an identical
    vector -- there is no model weight or randomness involved. This is a SPARSE, lexical
    representation: it only catches relevance through shared words."""
    vector = [0.0] * VECTOR_SIZE          # -> list[float], len 64, starts all zero
    words = re.findall(r"[a-z0-9]+", text.lower())
    for word, count in Counter(words).items():
        bucket = int(hashlib.md5(word.encode()).hexdigest(), 16) % VECTOR_SIZE
        vector[bucket] += count            # collisions just add counts into the same bucket
    return vector

def cosine_similarity(a: list[float], b: list[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = sum(x * x for x in a) ** 0.5
    norm_b = sum(y * y for y in b) ** 0.5
    return dot / (norm_a * norm_b) if norm_a and norm_b else 0.0  # -> float in [-1.0, 1.0]

def rerank_results(query: str, results: list[SearchResult], top_k: int = 2) -> list[SearchResult]:
    """Score each SearchResult (title + snippet) against the query, then keep only the
    top_k most relevant ones -- the rest never make it into the LLM's context."""
    query_vec = embed_text(query)
    scored = [
        (cosine_similarity(query_vec, embed_text(r.title + " " + r.snippet)), r)
        for r in results
    ]  # -> list[tuple[float, SearchResult]], same length as `results`
    scored.sort(key=lambda pair: pair[0], reverse=True)
    return [r for _, r in scored[:top_k]]  # -> list[SearchResult], len == top_k

In [ ]:
query = "starter hydration ratio"

# Raw results as a search API might return them: relevance mixed with popularity.
raw_results = [
    SearchResult("History of Sourdough Bread", "This ancient bread-making technique dates back thousands of years to Ancient Egypt.", "https://example-bake.org/history"),
    SearchResult("Hydration Ratio for Sourdough Starter", "A 100 percent hydration starter uses equal weights of flour and water by mass.", "https://example-bake.org/hydration"),
    SearchResult("Best Bread Knives 2026", "A serrated knife makes cleaner slices through a crusty loaf without tearing it.", "https://example-bake.org/knives"),
    SearchResult("Adjusting Starter Hydration for Climate", "Lower hydration starter mixtures ferment more slowly in humid kitchens.", "https://example-bake.org/climate-hydration"),
    SearchResult("Sourdough Discard Recipes", "Use leftover starter portions in pancakes or crackers instead of discarding them.", "https://example-bake.org/discard"),
]

print("BEFORE (raw API order):")
for r in raw_results:
    print(f"  - {r.title}")

print("\nAFTER (hash-embedding re-ranked, top 2):")
for r in rerank_results(query, raw_results, top_k=2):
    print(f"  - {r.title}")

### The same rerank with real, verifiable scores (TF-IDF via scikit-learn)

The hash-bucket embedding above is deliberately minimal. `TfidfVectorizer` is a genuine,
widely-used sparse embedding, and lets us print real cosine-similarity numbers instead of
just an order. Verified output is shown as a comment after the cell.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine_similarity
import numpy as np

titles = [r.title for r in raw_results]
docs = [f"{r.title}. {r.snippet}" for r in raw_results]

# fit_transform learns the corpus vocabulary AND encodes every doc in one call.
# shape: (n_docs=5, vocab_size) -- vocab_size depends on the corpus text, unlike the fixed
# VECTOR_SIZE=64 used by the hash-based embed_text above.
vectorizer = TfidfVectorizer(stop_words="english")
doc_matrix = vectorizer.fit_transform(docs)

# transform (NOT fit_transform) reuses that same fitted vocabulary for the query, so query
# and docs land in the exact same vector space and are directly comparable.
# shape: (1, vocab_size)
query_vec = vectorizer.transform([query])

sims = sk_cosine_similarity(query_vec, doc_matrix)[0]  # -> np.ndarray, shape (5,), one score per doc
order = np.argsort(-sims)                               # descending relevance

print(f"doc_matrix.shape = {doc_matrix.shape}, query_vec.shape = {query_vec.shape}")
print("\nAFTER (TF-IDF cosine rerank, real scores):")
for idx in order:
    print(f"  {sims[idx]:.4f}  {titles[idx]}")

# Verified output (this cell was run in this sandbox):
#   doc_matrix.shape = (5, 49), query_vec.shape = (1, 49)
#   0.5933  Hydration Ratio for Sourdough Starter
#   0.4310  Adjusting Starter Hydration for Climate
#   0.0984  Sourdough Discard Recipes
#   0.0000  History of Sourdough Bread
#   0.0000  Best Bread Knives 2026
# The two on-topic results score highest; the knife and history results score EXACTLY
# 0.0 because after stop-word removal they share zero vocabulary with the query --
# cosine similarity between non-overlapping vectors is mathematically zero, not just low.

### Dense embeddings (not executable in this sandbox)

TF-IDF is still lexical: it cannot connect "starter hydration ratio" to a result phrased as
"flour-to-water proportions" with zero shared words. A real dense embedding model closes that
gap by encoding meaning rather than counting tokens. This cell is written against the current,
real `sentence-transformers` API but is **not run** here -- `torch`/`transformers`/
`sentence-transformers` are not installed in this sandbox (see the note at the top of this
notebook).

In [ ]:
# NOT EXECUTED in this sandbox -- see the note at the top of this notebook.
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")  # loads a small pretrained transformer once

def embed_text_dense(texts: list[str]) -> np.ndarray:
    # normalize_embeddings=True returns unit vectors, so a plain dot product already
    # equals cosine similarity -- no separate norm division needed downstream.
    return model.encode(texts, normalize_embeddings=True)  # -> shape (len(texts), 384)

# query_vec = embed_text_dense([query])[0]           # shape (384,)
# doc_vecs = embed_text_dense(docs)                    # shape (5, 384)
# dense_sims = doc_vecs @ query_vec                    # unit vectors -> dot product == cosine similarity

## Day 3: Search + LLM Grounded Research

Grounding means the LLM answers only from retrieved material, cites which source backs each
claim, and admits when the material is insufficient rather than guessing. The pipeline below
chains a search step, a re-ranking step, and a (mock) grounded LLM call.

In [ ]:
def build_grounded_prompt(question: str, bundle: str) -> str:
    return (
        "You are a research assistant. Answer using ONLY the material below.\n\n"
        f"MATERIAL:\n{bundle}\n\n"
        f"QUESTION: {question}\n\n"
        "Rules:\n"
        "- Cite the source number [n] after every factual claim.\n"
        "- If the material does not answer the question, say "
        "'Not enough information in the provided sources.'\n"
        "- Do not add outside knowledge.\n"
    )

def call_llm_stub(prompt: str) -> str:
    """Mock LLM call. A real client (e.g. an Anthropic Messages API call) would send `prompt`
    to a hosted model and return its text response instead of this canned reply."""
    return (
        "QUIC is now supported by default in several major browsers [1]. Adoption on the server "
        "side is growing but remains behind HTTP/2 in raw traffic share [2]. Not enough information "
        "in the provided sources to say when server-side adoption will overtake HTTP/2.\n\n"
        "Sources:\n[1] https://example-net.org/quic-browsers\n[2] https://example-net.org/quic-traffic-share"
    )

In [ ]:
protocol_index = {
    "quic protocol adoption": [
        SearchResult(
            "QUIC Support Across Browsers",
            "QUIC is enabled by default in several major browsers as of this year.",
            "https://example-net.org/quic-browsers",
        ),
        SearchResult(
            "Server-Side Traffic Share Report",
            "QUIC traffic is rising but still trails HTTP/2 in overall share among measured servers.",
            "https://example-net.org/quic-traffic-share",
        ),
    ],
}

def search_web_v2(query: str, max_results: int = 5) -> list[SearchResult]:
    return protocol_index.get(query.lower().strip(), [])[:max_results]

def research(question: str, searcher, ranker, summarizer) -> str:
    """Chain: search -> re-rank -> bundle with sources -> grounded LLM summary.
    searcher/ranker/summarizer are passed in as arguments (not hardcoded) so a mock stub
    or a real client can be swapped in without touching this function's body at all."""
    raw_hits = searcher(question)                                        # -> list[SearchResult]
    top_hits = ranker(question, raw_hits, top_k=2) if raw_hits else raw_hits  # -> list[SearchResult]
    bundle = build_context_bundle(top_hits)                                # -> str
    prompt = build_grounded_prompt(question, bundle)                        # -> str
    return summarizer(prompt)                                              # -> str

answer = research(
    "quic protocol adoption",
    searcher=search_web_v2,
    ranker=rerank_results,
    summarizer=call_llm_stub,
)
print(answer)

### Validating citations in code

The prompt asks nicely; this function checks. It catches two specific failure modes: an
**uncited claim** (a sentence with no `[n]` marker) and a **hallucinated citation** (an `[n]`
pointing past the number of sources actually retrieved).

In [ ]:
import re

REFUSAL_PHRASE = "not enough information in the provided sources"

def split_claim_sentences(answer_body: str) -> list[str]:
    """Naive sentence split on end punctuation -- good enough for short grounded answers;
    a production version would use a real sentence tokenizer for edge cases like 'e.g.'"""
    raw = re.split(r"(?<=[.!?])\s+", answer_body.strip())
    return [s.strip() for s in raw if s.strip()]  # -> list[str], one entry per sentence

def validate_citations(answer: str, num_sources: int) -> dict:
    """Returns a report dict instead of raising, so callers decide how strict to be
    (e.g. silently drop an uncited sentence vs. reject the whole answer and retry)."""
    body = answer.split("Sources:")[0].strip()
    lower_body = body.lower()

    # The one sentence explicitly allowed to skip citation: the grounded refusal itself.
    if REFUSAL_PHRASE in lower_body and len(split_claim_sentences(body)) == 1:
        return {"ok": True, "uncited_claims": [], "out_of_range_citations": []}

    sentences = split_claim_sentences(body)                          # -> list[str], len = n_claims
    uncited = [s for s in sentences if not re.search(r"\[\d+\]", s)]  # -> list[str], failure mode 1

    cited_numbers = {int(n) for n in re.findall(r"\[(\d+)\]", body)}  # -> set[int]
    out_of_range = sorted(n for n in cited_numbers if n < 1 or n > num_sources)  # failure mode 2

    return {
        "ok": not uncited and not out_of_range,
        "uncited_claims": uncited,
        "out_of_range_citations": out_of_range,
    }

# --- test cases ---
good = (
    "The library added native retry support in v2.3 [1]. Backoff intervals are "
    "configurable via a backoff_factor argument [2].\n\n"
    "Sources:\n[1] https://example.dev/changelog/v2.3\n[2] https://example.dev/docs/retries"
)
hallucinated_citation = (
    "The library added native retry support in v2.3 [1]. It also supports circuit "
    "breakers out of the box [3].\n\n"
    "Sources:\n[1] https://example.dev/changelog/v2.3\n[2] https://example.dev/docs/retries"
)
uncited_claim = (
    "The library added native retry support in v2.3 [1]. It is the fastest retry "
    "library available today.\n\n"
    "Sources:\n[1] https://example.dev/changelog/v2.3\n[2] https://example.dev/docs/retries"
)
refusal = "Not enough information in the provided sources.\n\nSources:\n[1] https://example.dev/changelog/v2.3"

for name, text, n in [
    ("good", good, 2),
    ("hallucinated_citation", hallucinated_citation, 2),
    ("uncited_claim", uncited_claim, 2),
    ("refusal", refusal, 1),
]:
    print(name, "->", validate_citations(text, n))

# Verified output (run in this sandbox):
# good                   -> {'ok': True,  'uncited_claims': [], 'out_of_range_citations': []}
# hallucinated_citation  -> {'ok': False, 'uncited_claims': [], 'out_of_range_citations': [3]}
# uncited_claim          -> {'ok': False, 'uncited_claims': ['It is the fastest retry library available today.'], 'out_of_range_citations': []}
# refusal                -> {'ok': True,  'uncited_claims': [], 'out_of_range_citations': []}

### Snippets vs. full pages

A search snippet is often a fragment chosen by the provider's highlighting logic, not
necessarily the sentence that answers the question. Fetching the full page can recover detail
the snippet cut off. Tested against a stable target (Python's own docs); wrapped in
`try/except` with a fixed fallback string so the cell still runs offline.

In [ ]:
def fetch_page_text(url: str, timeout: float = 10.0) -> str:
    """Fetch and flatten a page's visible paragraph text. Falls back to an empty string
    on any network error so a failed fetch degrades to 'snippet only' instead of crashing
    the whole pipeline -- the same fallback philosophy as Day 1's mock search client."""
    try:
        resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=timeout)
        resp.raise_for_status()
    except requests.RequestException:
        return ""  # caller should treat this the same as "no full-page text available"
    soup = BeautifulSoup(resp.text, "html.parser")
    main = soup.find("main") or soup
    paragraphs = main.find_all("p")
    return " ".join(p.get_text(" ", strip=True) for p in paragraphs)  # -> str, flattened body text

_FETCH_FALLBACK = (
    "...cancelled. Example: Changed in version 3.7: When aw is cancelled due to a timeout, "
    "wait_for waits for aw to be cancelled. Previously, it raised TimeoutError immediately. "
    "Changed in version 3.10: Removed the loop parameter. Changed in version 3.11: Raises "
    "TimeoutError instead of asyncio.TimeoutError. Changed in version 3.12: Implemented using "
    "asyncio.timeout()..."
)

page_text = fetch_page_text("https://docs.python.org/3/library/asyncio-task.html")
if not page_text:
    page_text = _FETCH_FALLBACK
    print("[FALLBACK -- network unavailable, using cached snippet]")

idx = page_text.lower().find("wait_for")
print(page_text[max(0, idx - 40): idx + 220] if idx >= 0 else page_text[:260])

## Day 4: AI Tool Evaluation Framework

Any specific tool name in this space goes stale fast, so the evaluation itself stays
framework-agnostic: score a candidate tool on cost, security, and approval-friction, then reuse
the Day 3 `research()` pipeline to keep the underlying facts current.

In [ ]:
from dataclasses import dataclass

@dataclass
class ToolEvaluation:
    name: str
    pricing_model: str
    hidden_usage_costs: bool
    trains_on_input_data: bool
    sso_supported: bool
    requires_security_review: bool
    notes: str = ""

    def risk_score(self) -> int:
        """Additive risk score, 0 = lowest risk. Weights (1-2) are deliberately coarse --
        the point is to separate 'clearly fine' from 'clearly not' fast, not to produce a
        precise number worth debating to two decimal places."""
        score = 0
        score += 2 if self.hidden_usage_costs else 0          # can silently blow a budget
        score += 2 if self.trains_on_input_data else 0        # your data leaves your control
        score += 1 if not self.sso_supported else 0           # weaker auth / offboarding story
        score += 1 if self.requires_security_review else 0    # friction, not risk, but slows adoption
        return score

    def quick_verdict(self) -> str:
        score = self.risk_score()
        if score == 0:
            return "green"
        if score >= 4:
            return "red"
        return "yellow"

candidates = [
    ToolEvaluation(
        name="note-taking assistant",
        pricing_model="per-seat monthly",
        hidden_usage_costs=False,
        trains_on_input_data=False,
        sso_supported=True,
        requires_security_review=False,
        notes="Opt-out training disclosed clearly in settings.",
    ),
    ToolEvaluation(
        name="spreadsheet copilot",
        pricing_model="usage-metered, billed to a connected API key",
        hidden_usage_costs=True,
        trains_on_input_data=True,
        sso_supported=False,
        requires_security_review=True,
        notes="Retention policy unclear; needs compliance sign-off before rollout.",
    ),
    ToolEvaluation(
        name="code review bot",
        pricing_model="free tier, usage-metered beyond 500 reviews/month",
        hidden_usage_costs=False,
        trains_on_input_data=True,
        sso_supported=True,
        requires_security_review=True,
        notes="Trains on input by default but SSO + opt-out available on request.",
    ),
]

for c in candidates:
    print(f"{c.name}: risk_score={c.risk_score()} verdict={c.quick_verdict()} | {c.notes}")

# Verified output (run in this sandbox):
# note-taking assistant: risk_score=0 verdict=green | Opt-out training disclosed clearly in settings.
# spreadsheet copilot: risk_score=6 verdict=red | Retention policy unclear; needs compliance sign-off before rollout.
# code review bot: risk_score=3 verdict=yellow | Trains on input by default but SSO + opt-out available on request.

In [ ]:
# Tie back to Day 3: periodically research what's changed for a given tool category
# so the cost/security/approval-friction scoring above doesn't go stale.
landscape_index = {
    "spreadsheet copilot pricing and security changes": [
        SearchResult(
            "Spreadsheet Copilot Updates Data Retention Policy",
            "The vendor now offers an enterprise tier with opt-out training and a signed DPA.",
            "https://example-tools.org/copilot-policy-update",
        ),
    ],
}

def search_web_v3(query: str, max_results: int = 5) -> list[SearchResult]:
    return landscape_index.get(query.lower().strip(), [])[:max_results]

landscape_update = research(
    "spreadsheet copilot pricing and security changes",
    searcher=search_web_v3,
    ranker=rerank_results,
    summarizer=call_llm_stub,
)
print(landscape_update)